In [1]:
# Load environment variables and verify the project setup.
import sys, types
from pathlib import Path

# --- RAGAS compatibility shim (must run before importing ragas anywhere) ---
# ragas 0.4.x does `from langchain_community.chat_models.vertexai import ChatVertexAI`,
# a submodule removed in langchain-community 0.4.x. ragas only imports the NAME (never
# instantiates it unless you use Vertex AI), so register a stub to satisfy the import —
# this avoids pulling the heavy langchain-google-vertexai package.
if "langchain_community.chat_models.vertexai" not in sys.modules:
    _shim = types.ModuleType("langchain_community.chat_models.vertexai")
    _shim.ChatVertexAI = type("ChatVertexAI", (), {})
    sys.modules["langchain_community.chat_models.vertexai"] = _shim

# Find the repo root (the folder containing env_checker.py) and make it importable.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "env_checker.py").exists())
sys.path.insert(0, str(ROOT))

# Load .env into the environment for this session.
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ModuleNotFoundError:
    print("python-dotenv not installed yet — run: uv add python-dotenv")

# Verify .env variables and required packages.
from env_checker import run_checks
run_checks()

Environment variables (from .env.example)
  ✓ OPENAI_API_KEY  — set
  ✓ ANTHROPIC_API_KEY  — set
  ✓ LANGSMITH_TRACING  — set
  ✓ LANGSMITH_ENDPOINT  — set
  ✓ LANGSMITH_API_KEY  — set
  ✓ LANGSMITH_PROJECT  — set
  ✓ CHROMA_PERSIST_DIR  — set

Required packages (from pyproject.toml)
  ✓ beautifulsoup4  — installed (4.14.3)
  ✓ chromadb  — installed (1.5.9)
  ✓ langchain  — installed (1.3.2)
  ✓ langchain-chroma  — installed (1.1.0)
  ✓ langchain-classic  — installed (1.0.7)
  ✓ langchain-community  — installed (0.4.2)
  ✓ langchain-core  — installed (1.4.0)
  ✓ langchain-experimental  — installed (0.4.2)
  ✓ langchain-openai  — installed (1.2.2)
  ✓ lxml  — installed (6.1.1)
  ✓ onnxruntime  — installed (1.19.2)
  ✓ pypdf  — installed (6.12.2)
  ✓ python-dotenv  — installed (1.2.2)
  ✓ ragas  — installed (0.4.3)
  ✓ rank-bm25  — installed (0.2.2)
  ✓ rapidfuzz  — installed (3.14.5)
  ✓ ipykernel  — installed (7.2.0)
  ✓ jupyterlab  — installed (4.5.7)

✓ All checks passed.


True

# Faithfulness

**Faithfulness** = the fraction of factual claims in the *answer* that are actually supported by the *retrieved context*. It catches hallucination: an answer that states things the context doesn't back up scores low.

We compute it as **LLM-as-judge** (the same approach RAGAS uses internally): split the answer into atomic claims, then verify each against the context.

> RAGAS has a built-in `Faithfulness` metric, but `ragas` currently imports a removed `langchain_community` path and won't load on this LangChain 1.x stack, so we implement the metric directly.

## End-to-end RAG over the example PDF

Generate a *real* answer to evaluate: load the SDLC PDF → semantic chunks → store embeddings in Chroma → LCEL chain to query & answer. The faithfulness section below can then score this generated `rag_answer` against `rag_context`.

### 1. Load the PDF

In [2]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = ROOT / "assets/sample-docs/sdlc-end-to-end.pdf"
docs = PyPDFLoader(str(pdf_path)).load()
print(f"Loaded {len(docs)} page(s)")

/tmp/claude-501/ipykernel_74984/4130999730.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 12 page(s)


### 2. Semantic chunking

In [3]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
chunks = SemanticChunker(
    embedding_model, breakpoint_threshold_type="percentile"
).split_documents(docs)
print(f"{len(docs)} pages -> {len(chunks)} semantic chunks")

/tmp/claude-501/ipykernel_74984/1029204723.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


12 pages -> 23 semantic chunks


### 3. Embed & store in Chroma

In [4]:
import os
from langchain_chroma import Chroma

persist_dir = str(ROOT / os.getenv("CHROMA_PERSIST_DIR", "./chroma_db"))
vector_store = Chroma(
    collection_name="faithfulness_demo",
    embedding_function=embedding_model,
    persist_directory=persist_dir,
)
vector_store.reset_collection()        # start clean so re-runs do not duplicate
vector_store.add_documents(chunks)
print("Stored", vector_store._collection.count(), "chunks in collection 'faithfulness_demo'")

Stored 23 chunks in collection 'faithfulness_demo'


### 4. LCEL chain — query & answer

`{context, question} | prompt | llm | parser`. We also capture the retrieved `rag_context` separately so faithfulness can check the answer against it.

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

retriever = vector_store.as_retriever(search_kwargs={"k": 4})

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

prompt = ChatPromptTemplate.from_template(
    "Answer the question using ONLY the context below.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

question = "What is Low-Level Design (LLD) and who is responsible for it?"
rag_context = format_docs(retriever.invoke(question))   # context the answer is grounded in
rag_answer = rag_chain.invoke(question)

print("Q:", question)
print("A:", rag_answer)

Q: What is Low-Level Design (LLD) and who is responsible for it?
A: Low-Level Design (LLD) provides detailed internal design of each component — the blueprint developers code from. It is owned by Senior Developers / Tech Lead.


> To score this generated answer with the faithfulness judge below, set `context, answer = rag_context, rag_answer` (replacing the hardcoded example).

## Faithfulness with RAGAS

RAGAS ships a built-in `Faithfulness` metric. Note: ragas 0.4.x imports a `langchain_community` Vertex AI submodule that was removed in LangChain 1.x — the **setup cell at the top** registers a small stub to satisfy that import, so no heavy Google packages are needed. We score the generated `rag_answer` against its retrieved contexts.

In [17]:
from ragas import SingleTurnSample
from ragas.metrics import Faithfulness
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI

eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
faithfulness_metric = Faithfulness(llm=eval_llm)

# Score the RAG answer (rag_answer) against the contexts it was retrieved from.
sample = SingleTurnSample(
    user_input=question,
    response=rag_answer,
    retrieved_contexts=[d.page_content for d in retriever.invoke(question)],
)

# Top-level await works in the Jupyter kernel.
score = await faithfulness_metric.single_turn_ascore(sample)
print(f"RAGAS faithfulness = {score:.3f}")

/Users/Prabhukumar/Projects/PycharmProjects/rag-reference/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/tmp/claude-501/ipykernel_74984/1501122998.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness
/tmp/claude-501/ipykernel_74984/1501122998.py:6: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))


RAGAS faithfulness = 0.750
RAGAS faithfulness = 1.000


## Example to evaluate

An answer generated from some retrieved context — note one unsupported claim.

In [18]:
context = (
    "Low-Level Design (LLD) provides the detailed internal design of each component — "
    "the blueprint developers code from. It specifies class/module structure, methods, "
    "data structures and database fields."
)
answer = (
    "LLD describes the detailed internal design of each component, including classes, "
    "methods and database fields. It is written by the QA team."   # <- unsupported
)

## Judge: claim extraction + verification

In [19]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

class ClaimVerdict(BaseModel):
    claim: str = Field(description="an atomic factual claim taken from the answer")
    supported: bool = Field(description="True if the claim can be inferred from the context")
    reason: str

class FaithfulnessReport(BaseModel):
    verdicts: list[ClaimVerdict]

judge = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(FaithfulnessReport)

PROMPT = """You evaluate the FAITHFULNESS of an answer against its retrieved context.
Step 1: split the ANSWER into atomic factual claims.
Step 2: for each claim, decide if it can be directly inferred from the CONTEXT.

CONTEXT:
{context}

ANSWER:
{answer}
"""

report = judge.invoke(PROMPT.format(context=context, answer=answer))

## Score

faithfulness = supported claims / total claims (1.0 = fully grounded).

In [20]:
for v in report.verdicts:
    print(("OK " if v.supported else "XX"), v.claim)

supported = sum(v.supported for v in report.verdicts)
faithfulness = supported / len(report.verdicts)
print(f"\nFaithfulness = {supported}/{len(report.verdicts)} = {faithfulness:.2f}")

OK  LLD describes the detailed internal design of each component.
OK  LLD includes classes, methods and database fields.
XX LLD is written by the QA team.

Faithfulness = 2/3 = 0.67
OK  LLD describes the detailed internal design of each component.
OK  LLD includes classes, methods and database fields.
XX LLD is written by the QA team.

Faithfulness = 2/3 = 0.67
